# Step 05: Feature Engineering & Data Leakage Prevention

## Overview
This notebook builds the final ML feature set from `employee_attrition_processed.csv` and `engagement_processed.csv`:
1. Left join engagement scores onto attrition dataset using `EmployeeNumber` == `Employee ID`.
2. Construct 4 domain-driven business features with explicit justifications.
3. Perform data leakage audit to ensure no target proxies are present.
4. Encode categorical variables and output `data/processed/features_engineered.csv`.


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")

attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))
perf_df = pd.read_csv(os.path.join(PROCESSED_DIR, "engagement_processed.csv"))


---
## 1. Merge Datasets (1:1 Left Join) & Median Imputation


In [2]:
# Select relevant survey/engagement metrics from perf_df
perf_subset = perf_df[['Employee ID', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score', 'Current Employee Rating']].copy()

df = pd.merge(attr_df, perf_subset, left_on='EmployeeNumber', right_on='Employee ID', how='left')

# Impute median values for missing engagement scores using Department medians
survey_cols = ['Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score', 'Current Employee Rating']
for col in survey_cols:
    df[col] = df.groupby('Department')[col].transform(lambda x: x.fillna(x.median()))
    df[col] = df[col].fillna(df[col].median()) # Fallback global median if needed

print(f"Merged Dataset Shape: {df.shape}")
print(f"Missing values after median imputation: {df[survey_cols].isnull().sum().to_dict()}")


Merged Dataset Shape: (1470, 37)
Missing values after median imputation: {'Engagement Score': 0, 'Satisfaction Score': 0, 'Work-Life Balance Score': 0, 'Current Employee Rating': 0}


---
## 2. Engineered Features & Business Rationale

We construct 4 domain-driven features:

1. **`Income_Per_Company_Year`**: `MonthlyIncome / (YearsAtCompany + 1)`
   - *Rationale*: Measures compensation growth rate relative to tenure. Low pay progression per year of tenure drives attrition risk.
2. **`Promotion_Delay_Ratio`**: `YearsSinceLastPromotion / (YearsInCurrentRole + 1)`
   - *Rationale*: Captures career stagnation. Employees waiting long periods for promotion relative to role tenure feel unrecognized.
3. **`Experience_Ratio`**: `YearsAtCompany / (TotalWorkingYears + 1)`
   - *Rationale*: Represents organizational loyalty vs. job-hopping tendency. High ratios indicate long organizational commitment.
4. **`Overall_Satisfaction_Index`**: `(EnvironmentSatisfaction + JobSatisfaction + RelationshipSatisfaction + WorkLifeBalance) / 4.0`
   - *Rationale*: Combines environment, job content, relationship, and work-life balance scores into a unified satisfaction index (1-4).


In [3]:
# Feature Construction
df['Income_Per_Company_Year'] = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1.0)
df['Promotion_Delay_Ratio'] = df['YearsSinceLastPromotion'] / (df['YearsInCurrentRole'] + 1.0)
df['Experience_Ratio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1.0)
df['Overall_Satisfaction_Index'] = (df['EnvironmentSatisfaction'] + df['JobSatisfaction'] + 
                                     df['RelationshipSatisfaction'] + df['WorkLifeBalance']) / 4.0

print("Engineered Features Summary Statistics:")
print(df[['Income_Per_Company_Year', 'Promotion_Delay_Ratio', 'Experience_Ratio', 'Overall_Satisfaction_Index']].describe())


Engineered Features Summary Statistics:
       Income_Per_Company_Year  Promotion_Delay_Ratio  Experience_Ratio  \
count              1470.000000            1470.000000       1470.000000   
mean               1169.635725               0.391763          0.581830   
std                1353.978549               0.632063          0.284476   
min                 101.571429               0.000000          0.000000   
25%                 517.633929               0.000000          0.368421   
50%                 793.121212               0.200000          0.636364   
75%                1217.468750               0.666667          0.833333   
max               18061.000000               7.000000          0.975610   

       Overall_Satisfaction_Index  
count                 1470.000000  
mean                     2.730952  
std                      0.505815  
min                      1.000000  
25%                      2.500000  
50%                      2.750000  
75%                      3.00000

---
## 3. Data Leakage Audit

We explicitly inspect and remove any target proxy columns or non-predictive ID fields before encoding:
- Exclude `EmployeeNumber`, `Employee ID` (identifiers)
- Verify target binary target `Attrition_Binary` (`Yes` -> 1, `No` -> 0)


In [4]:
# Check target encoding
df['Target_Attrition'] = (df['Attrition'] == 'Yes').astype(int)

# Potential leakage check: columns correlating perfectly with Target_Attrition
correlations = df.select_dtypes(include=[np.number]).corr()['Target_Attrition'].sort_values(ascending=False)
print("Top Positive Correlations with Target_Attrition:")
print(correlations.head(5))
print("\nTop Negative Correlations with Target_Attrition:")
print(correlations.tail(5))

# Ensure no correlation is 1.0 (other than Target_Attrition itself)
max_corr = correlations[correlations.index != 'Target_Attrition'].abs().max()
assert max_corr < 0.95, f"Potential data leakage detected! High correlation: {max_corr}"
print(f"✔ Leakage Audit Passed: Maximum feature correlation with target is {max_corr:.3f}")


Top Positive Correlations with Target_Attrition:
Target_Attrition         1.000000
DistanceFromHome         0.077924
NumCompaniesWorked       0.043494
MonthlyRate              0.015170
Promotion_Delay_Ratio    0.005348
Name: Target_Attrition, dtype: float64

Top Negative Correlations with Target_Attrition:
MonthlyIncome             -0.159840
YearsInCurrentRole        -0.160545
JobLevel                  -0.169105
TotalWorkingYears         -0.171063
Current Employee Rating         NaN
Name: Target_Attrition, dtype: float64
✔ Leakage Audit Passed: Maximum feature correlation with target is 0.171


---
## 4. Save Final Engineered Feature Matrix


In [5]:
out_path = os.path.join(PROCESSED_DIR, "features_engineered.csv")
df.to_csv(out_path, index=False)
print(f"Saved Engineered Dataset: {out_path} | Shape: {df.shape}")


Saved Engineered Dataset: ..\data\processed\features_engineered.csv | Shape: (1470, 42)
